[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-boosting.ipynb)

# Boosting — AdaBoost, Gradient Boosting & XGBoost

*AIBits Academy · Machine Learning End To End · Ensemble Learning · New*

Where Random Forest builds independent trees in parallel and averages them, boosting builds trees sequentially — each one correcting the mistakes of everything built before it.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **🎯 Intuition First**
>
> Imagine hiring a team of specialists one at a time, where every new hire is chosen specifically to fix the mistakes the current team is making. The first person can be a modest generalist. The second is deliberately picked to be strong on exactly the cases where person one struggles. The third is picked to help with what one-plus-two still get wrong. Each new hire is weak on its own, but the team as a whole gets progressively harder to fool. Boosting is exactly this hiring strategy — applied to shallow decision trees, one after another — and it turns out to be one of the single most reliable ways to squeeze accuracy out of tabular data. That's why XGBoost/LightGBM/CatBoost win so many real-world tabular competitions.

> **📋 Real-World Case Study — Emotion Detection from Speech**
>
> A genuinely striking ensemble application: extracting acoustic features from audio clips (pitch variance, energy, spectral characteristics, tempo) and boosting a classifier to detect the speaker's emotional state (happy, angry, neutral, sad). Call-centre quality monitoring uses exactly this — automatically flagging calls where customer frustration is escalating, feeding an ensemble classifier that combines many weak acoustic-feature-based rules into a single, well-calibrated escalation signal.

## Bagging vs Boosting — The Core Contrast

|  | Bagging (Random Forest) | Boosting (AdaBoost / GBM / XGBoost) |
|---|---|---|
| Tree construction | Parallel, independent | Sequential — each tree depends on the previous |
| Primary goal | Reduce **variance** | Reduce **bias** (and variance, via shrinkage) |
| Base learners | Deep, low-bias trees | Shallow, high-bias "weak learners" (stumps, depth 3–6) |
| Overfitting risk | Low — more trees rarely hurt | Higher — too many rounds can overfit; needs early stopping |
| Sensitivity to noise/outliers | Robust | More sensitive — misclassified noisy points get up-weighted |

## AdaBoost — Adaptive Boosting

AdaBoost fits a sequence of weak learners (typically decision stumps — depth-1 trees), where each new learner focuses on the samples the previous ensemble got wrong, by re-weighting the training data.

$$\begin{gathered}\text{Weighted error:}\quad \varepsilon_m = \dfrac{\sum_i w_i\cdot \mathbb{1}(y_i\neq h_m(x_i))}{\sum_i w_i}\\[8pt]\text{Learner weight:}\quad \alpha_m = \tfrac{1}{2}\ln\!\left(\dfrac{1-\varepsilon_m}{\varepsilon_m}\right)\\[8pt]\text{Sample re-weight:}\quad w_i \leftarrow w_i\cdot \exp(-\alpha_m y_i h_m(x_i)) \ \ (\text{then renormalise } \textstyle\sum w_i=1)\end{gathered}$$

Intuition: a learner that does *better than random* (εₘ < 0.5) gets a positive weight αₘ — the better it is, the more it counts in the final vote. Misclassified samples get their weight multiplied up (exp of a positive number), so the next weak learner is forced to pay more attention to exactly the points the ensemble is currently getting wrong.

## Worked Example — 3 Rounds by Hand

A toy dataset of 5 Surat textile-QC samples (pass=+1, fail=−1), all starting with equal weight 0.2:

| Round | Weak learner (stump) | Weighted error εₘ | αₘ = ½ln((1−ε)/ε) | Effect |
|---|---|---|---|---|
| 1 | Split on thread_count > 150 | 0.20 | 0.693 | Misclassified sample's weight ×e^0.693 ≈ ×2.0 |
| 2 | Split on strength > 120N | 0.27<sup>*</sup> | 0.496 | Now-harder sample re-weighted again; different sample misclassified this round |
| 3 | Split on defect_count ≤ 2 | 0.14<sup>*</sup> | 0.906 | High confidence — this stump gets a large vote in the final ensemble |

* computed against the *updated, renormalised* weights from the previous round, not the raw counts.

$$\text{Final prediction:}\quad H(x) = \operatorname{sign}\Big(\sum_m \alpha_m\cdot h_m(x)\Big)$$

The final classification is a **weighted vote** across all weak learners — not a simple majority. A confident, low-error stump (large α) can outvote several weaker ones.

## Gradient Boosting — A More General View

AdaBoost re-weights samples using an exponential loss. Gradient Boosting generalises this: at each stage, fit a new weak learner to the **negative gradient (residuals)** of an arbitrary differentiable loss function with respect to the current ensemble's predictions.

$$\begin{gathered}F_0(x) = \operatorname*{argmin}_{c}\sum_i L(y_i,c) \ \ (\text{initial constant prediction})\\[8pt]r_{im} = -\left[\dfrac{\partial L(y_i,F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}} \ \ (\text{pseudo-residuals})\\[8pt]F_m(x) = F_{m-1}(x) + \nu\cdot h_m(x) \ \ (h_m \text{ fit to predict } r_{im};\ \nu = \text{learning rate / shrinkage})\end{gathered}$$

For squared-error loss, the pseudo-residual rᵢₘ is exactly (yᵢ − Fₘ₋₁(xᵢ)) — literally the current prediction error. Each new tree is trained to predict *what the ensemble is still getting wrong*, and its contribution is shrunk by ν (typically 0.01–0.3) so no single tree dominates — this shrinkage is the main lever against overfitting in boosting.

> **📊 Prerequisite refresher**
>
> This is the same derivative-as-direction-of-improvement idea from the **Calculus for ML** prerequisite page, just one level more abstract: instead of a gradient with respect to a fixed parameter vector θ (Linear/Logistic Regression), Gradient Boosting takes it with respect to the ensemble's own predictions F(x) — a "gradient descent in function space." The pseudo-residual rᵢₘ is nothing more than −∂L/∂F(xᵢ), the same downhill-pointing logic, with the model's output standing in for the parameter.

## XGBoost — Gradient Boosting, Engineered

XGBoost (and its cousins LightGBM, CatBoost) add: a regularization term directly in the tree-building objective (penalising leaf count and leaf-weight magnitude — effectively baking L1/L2 regularization into the trees themselves), second-order (Newton) gradient information for faster, more accurate splits, built-in handling of missing values, and heavy engineering for speed (histogram binning, parallel/GPU split-finding).

## Sequential Correction — Visualised

## Code — Comparing AdaBoost, Gradient Boosting & XGBoost

Predicting whether a Flipkart product listing will be clicked (CTR classification):

In [ ]:
import numpy as np
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# Flipkart listing CTR dataset (synthetic)
np.random.seed(7)
n = 3000
price_rank   = np.random.uniform(0,1,n)     # 0=cheapest in category, 1=priciest
rating       = np.random.uniform(2,5,n)
num_images   = np.random.randint(1,8,n)
discount_pct = np.random.uniform(0,70,n)
logit = (-2*price_rank + 0.9*(rating-3.5) + 0.15*num_images
         + 0.03*discount_pct + np.random.normal(0,0.6,n))
y = (logit > np.median(logit)).astype(int)
X = np.column_stack([price_rank, rating, num_images, discount_pct])

X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # decision stumps
    n_estimators=200, learning_rate=0.8, random_state=42)
gbm = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=3,
    subsample=0.8, random_state=42)
xgbc = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    eval_metric='logloss', random_state=42)

for name, model in [('AdaBoost',ada), ('GradientBoosting',gbm), ('XGBoost',xgbc)]:
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
    print(f"{name:18s} Test ROC-AUC = {auc:.4f}")

## HistGradientBoosting — Boosting That Handles Missing Values Natively

Real-world data is rarely complete — Ahmedabad property listings routinely omit a field like the building's age when a seller doesn't know or report it. Classic `GradientBoostingRegressor` refuses to fit on data containing NaN at all; `HistGradientBoostingRegressor` (scikit-learn's histogram-based, LightGBM-inspired implementation) handles missing values natively by learning, at each split, which branch a missing value should default to.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
n = 500
area_sqft   = np.random.uniform(500, 3000, n)
age_years   = np.random.uniform(0, 30, n)
distance_km = np.random.uniform(1, 25, n)               # from Ahmedabad city centre
price_lakh  = 15 + 0.045*area_sqft - 0.9*age_years - 1.2*distance_km + np.random.normal(0, 8, n)

df = pd.DataFrame({'area_sqft': area_sqft, 'age_years': age_years,
                    'distance_km': distance_km, 'price_lakh': price_lakh})

# Simulate 15% unreported building age — realistic for resale listings
mask = np.random.RandomState(1).rand(n) < 0.15
df.loc[mask, 'age_years'] = np.nan

X = df[['area_sqft', 'age_years', 'distance_km']]
y = df['price_lakh']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

hgb = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
pred = hgb.predict(X_test)
print(f"HistGradientBoosting  MSE: {mean_squared_error(y_test, pred):.2f}  "
      f"R2: {r2_score(y_test, pred):.4f}")

try:
    GradientBoostingRegressor(random_state=42).fit(X_train, y_train)
except ValueError as e:
    print(f"GradientBoostingRegressor error: {e}")

With 15% of `age_years` unreported, `GradientBoostingRegressor.fit()` raises immediately — before you even find out whether the model would have been any good, you're forced into a separate imputation step. `HistGradientBoostingRegressor` trains directly on the same data as-is, reaching R²=0.9353 on the held-out test set with no imputation code at all.

> **🔗 When To Reach For This**
>
> XGBoost (covered above) also handles missing values natively and is usually the stronger choice for competition-grade tuning. `HistGradientBoostingRegressor`'s advantage is that it ships inside scikit-learn itself — no extra dependency — while matching XGBoost's speed on mid-sized tabular data via the same histogram-binning trick.

## Key Hyperparameters

| Parameter | Typical range | Effect |
|---|---|---|
| n_estimators | 100–1000 | More rounds → lower bias, higher overfitting risk without shrinkage/early stopping |
| learning_rate (ν) | 0.01–0.3 | Smaller → needs more trees but generalises better ("shrinkage") |
| max_depth | 3–8 | Boosting trees are intentionally shallow (weak learners); deep trees overfit fast |
| subsample | 0.6–1.0 | Row sub-sampling per round (stochastic gradient boosting) — reduces variance |
| colsample_bytree (XGBoost) | 0.5–1.0 | Column sub-sampling per tree — like Random Forest's feature bagging, reduces correlation between trees |
| reg_lambda / reg_alpha (XGBoost) | 0–10 | L2 / L1 penalty on leaf weights, directly inside the tree objective |

> **⚠ Boosting Needs Early Stopping**
>
> Unlike Random Forest (where adding more trees essentially never hurts), boosting *will* overfit if you run too many rounds — the ensemble keeps chasing residuals down to noise. Always monitor a validation set and use `early_stopping_rounds` (XGBoost/LightGBM) or `n_iter_no_change` (sklearn's GradientBoostingClassifier) to stop automatically.

## Stacking — Combining Heterogeneous Models via a Meta-Learner

Bagging and boosting both combine many versions of the *same type* of weak learner (usually trees). Stacking takes a different approach: train several genuinely different model types (say, Logistic Regression, Random Forest, and SVM) on the same data, then train a **meta-learner** to combine their predictions — letting the meta-learner discover which base model to trust in which situations.

$$\hat{y}_{\text{stack}} = g\big(f_1(x), f_2(x),\ldots,f_K(x)\big) \quad \text{where } f_1\ldots f_K \text{ are base learners, } g \text{ is the meta-learner}$$

> **⚠ Base Learners Must Predict on Held-Out Folds**
>
> If the meta-learner is trained on the base learners' predictions on the *same* data the base learners were fit on, it will over-trust base learners that simply overfit their training data. The standard fix — used by `StackingClassifier` internally — is to generate each base learner's training-time predictions via cross-validation (out-of-fold predictions), exactly like the nested-CV discipline from the Hyperparameter Tuning chapter.

In [ ]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

base_learners = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('svm', SVC(probability=True, random_state=42)),
]
# meta-learner: a simple model is usually enough — it only combines 3 numbers, not raw features
stack = StackingClassifier(estimators=base_learners,
                            final_estimator=LogisticRegression(),
                            cv=5)  # 5-fold out-of-fold predictions feed the meta-learner

scores = cross_val_score(stack, X_tr, y_tr, cv=5, scoring='roc_auc')
print(f"Stacked ensemble CV ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

Stacking tends to help most when the base learners make genuinely *different kinds* of mistakes — a linear model and a tree-based model err on different regions of the input space, giving the meta-learner real signal to exploit. Stacking three variants of the same Random Forest with different seeds gives the meta-learner almost nothing to work with, since their errors are highly correlated.

## Voting Classifiers — Combining Predictions Without a Meta-Learner

Stacking trains a meta-learner on top of the base learners' predictions. A **Voting Classifier** is the simpler cousin: it combines predictions directly, with no meta-learner to train at all. **Hard voting** takes a plain majority vote across base learners' predicted classes. **Soft voting** instead averages each base learner's predicted *class probabilities* and picks the class with the highest average — using more information per learner than a single hard vote, at the cost of requiring every base learner to support `predict_proba`.

In [ ]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

# Three genuinely different model families — a linear model, a tree ensemble, a probabilistic model
clf1 = LogisticRegression(max_iter=1000, random_state=4)
clf2 = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=4)
clf3 = GaussianNB()

for name, clf in [('Logistic Regression',clf1), ('Random Forest',clf2), ('Naive Bayes',clf3)]:
    clf.fit(X_tr, y_tr)
    print(f"{name:20s} {clf.score(X_te, y_te):.4f}")

hard = VotingClassifier(estimators=[('lr',clf1),('rf',clf2),('nb',clf3)], voting='hard').fit(X_tr, y_tr)
soft = VotingClassifier(estimators=[('lr',clf1),('rf',clf2),('nb',clf3)], voting='soft').fit(X_tr, y_tr)
print(f"{'Hard Voting':20s} {hard.score(X_te, y_te):.4f}")
print(f"{'Soft Voting':20s} {soft.score(X_te, y_te):.4f}")

Soft voting (0.8800) beats every individual base learner here, including the best one (Random Forest, 0.8667) — averaging class probabilities lets a base learner that's only *moderately* confident about a prediction be outweighed by others that are more confident, information a hard vote throws away entirely. Hard voting only ties the best individual model in this run, since a majority vote among 3 learners has no mechanism to weigh a confident prediction more than a marginal one.

> **⚠ Voting Isn't Automatically Better**
>
> Voting helps most when base learners are reasonably accurate individually *and* make different kinds of mistakes. Adding a distinctly weak base learner can drag hard voting down, since every vote counts equally regardless of that learner's track record — this is precisely the scenario where Stacking's trained meta-learner has an advantage: it can learn to down-weight a consistently unreliable base learner, something a plain vote cannot do.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Gradient boosting baseline

Fit `GradientBoostingClassifier(random_state=0)` on the training data and store the test ROC-AUC in `auc`.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
X, y = make_classification(n_samples=1500, n_features=10, n_informative=5, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
auc = None   # TODO


In [ ]:
try:
    check("AUC above 0.9", auc > 0.9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
X, y = make_classification(n_samples=1500, n_features=10, n_informative=5, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
gb = GradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
auc = roc_auc_score(y_te, gb.predict_proba(X_te)[:, 1])

```

</details>

### Exercise 2 · Medium · More trees, better fit?

Compare `n_estimators=5` against `n_estimators=200` (learning_rate 0.1). Store the two test AUCs in `auc_5` and `auc_200`, and `more_is_better` = whether the larger ensemble wins.

In [ ]:
auc_5 = auc_200 = more_is_better = None   # TODO (reuse the split from exercise 1)


In [ ]:
try:
    check("both computed", auc_5 is not None and auc_200 is not None)
    check("200 trees beat 5", more_is_better is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def auc_with(n):
    m = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, random_state=0).fit(X_tr, y_tr)
    return roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])
auc_5, auc_200 = auc_with(5), auc_with(200)
more_is_better = bool(auc_200 > auc_5)

```

</details>

### Exercise 3 · Stretch · One AdaBoost re-weighting step

Write `adaboost_update(w, wrong)`: given sample weights `w` (sum 1) and a boolean array `wrong`, compute the weighted error `err`, `alpha = 0.5*ln((1-err)/err)`, multiply the weights of wrong samples by `exp(alpha)` and the others by `exp(-alpha)`, renormalise, and return `(new_w, alpha)`.

In [ ]:
import numpy as np
def adaboost_update(w, wrong):
    pass   # TODO


In [ ]:
try:
    w = np.full(5, 0.2)
    new_w, alpha = adaboost_update(w, np.array([False, False, True, False, False]))
    check("alpha = 0.5 ln 4", abs(alpha - 0.5 * np.log(4)) < 1e-9)
    check("weights renormalised", abs(new_w.sum() - 1) < 1e-9)
    check("the mistake now carries half the weight", abs(new_w[2] - 0.5) < 1e-9 and abs(new_w[0] - 0.125) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def adaboost_update(w, wrong):
    err = w[wrong].sum()
    alpha = 0.5 * np.log((1 - err) / err)
    new_w = w * np.exp(np.where(wrong, alpha, -alpha))
    return new_w / new_w.sum(), alpha

```

Mistakes get heavier every round, so the next weak learner is forced to focus on them.

</details>

---
*Back to the course: **Machine Learning End To End → Boosting — AdaBoost, Gradient Boosting & XGBoost**.*